# 02 - Limpieza y preprocesamiento

Este notebook aplica reglas de limpieza **trazables y reproducibles** sobre el dataset original.

## Principios
1. No inventar información que no puede reconstruirse con evidencia.
2. No eliminar filas masivamente por valores faltantes.
3. Separar **limpieza de calidad** de **imputación para Machine Learning**.
4. Conservar indicadores (`flags`) de anomalías cuando puedan aportar información.
5. Evitar usar la variable objetivo `churn` para decidir cómo imputar predictores.

> La imputación estadística definitiva para los modelos se hará dentro del pipeline de entrenamiento, aprendiendo parámetros únicamente sobre el conjunto de entrenamiento.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_PATH = Path('../data/raw/sales_marketing_customer_raw.csv')
OUTPUT_PATH = Path('../data/processed/customer_clean.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f'Filas originales: {len(df):,}')
print(f'Columnas originales: {df.shape[1]}')


## 1. Reglas de limpieza

| Variable / problema | Regla |
|---|---|
| `customer_id` | Se conserva para trazabilidad, pero no será predictor. |
| `age` faltante o < 18 | Se marca como faltante. No se imputa aquí. |
| `gender` faltante | Se reemplaza por `Unknown`, categoría explícita. |
| `coupon_code` faltante | Se reemplaza por `NO_COUPON`; la ausencia de código es una categoría observable. |
| `total_spent` faltante | Se conserva como NaN y se crea un indicador de faltante. |
| `satisfaction_score` faltante | Se conserva como NaN y se crea un indicador de faltante. |
| Fechas | Se convierten a `datetime`; errores de parseo pasan a NaT. |
| `last_purchase_date < signup_date` | No se inventa una fecha. Se marca inconsistencia y la última compra se invalida para cálculos temporales. |
| `country` / `city` incoherentes | No se corrigen sin fuente externa confiable. Se conservarán para EDA y se evaluará excluir `city` del modelo. |


In [ ]:
# Conversión segura de fechas
date_cols = ['signup_date', 'last_purchase_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Fechas no parseables:')
display(df[date_cols].isna().sum().to_frame('cantidad'))


## 2. Edad: valores inválidos y faltantes

In [ ]:
df['age_invalid_flag'] = (
    df['age'].notna() & (df['age'] < 18)
).astype('int8')

age_invalid_count = int(df['age_invalid_flag'].sum())
age_missing_original = int(df['age'].isna().sum())

df.loc[df['age_invalid_flag'].eq(1), 'age'] = np.nan

print(f'Edad faltante original: {age_missing_original:,}')
print(f'Edad < 18 invalidada: {age_invalid_count:,}')
print(f'Edad faltante luego de la regla: {df["age"].isna().sum():,}')
print(f'Rango válido resultante: {df["age"].min():.0f} - {df["age"].max():.0f}')


**Decisión:** no imputamos la edad todavía. Para modelado, la mediana deberá aprenderse exclusivamente con `X_train` dentro de un `Pipeline`, evitando contaminación entre train y test.

## 3. Variables categóricas faltantes

In [ ]:
df['gender_missing_flag'] = df['gender'].isna().astype('int8')
df['gender'] = df['gender'].fillna('Unknown')

df['coupon_missing_flag'] = df['coupon_code'].isna().astype('int8')
df['coupon_code'] = df['coupon_code'].fillna('NO_COUPON')

display(pd.DataFrame({'gender': df['gender'].value_counts(dropna=False)}).head(10))
display(df['coupon_code'].value_counts(dropna=False).to_frame('cantidad'))


## 4. Variables numéricas faltantes

In [ ]:
for col in ['total_spent', 'satisfaction_score']:
    df[f'{col}_missing_flag'] = df[col].isna().astype('int8')

missing_numeric = df[['total_spent', 'satisfaction_score']].isna().sum().to_frame('faltantes')
display(missing_numeric)


No se rellenan todavía `total_spent` ni `satisfaction_score`. Hacerlo ahora con la mediana global haría que el conjunto de test influya indirectamente en la transformación. La imputación será parte del pipeline de Machine Learning.

## 5. Consistencia temporal

In [ ]:
df['temporal_inconsistency_flag'] = (
    df['last_purchase_date'].notna()
    & df['signup_date'].notna()
    & (df['last_purchase_date'] < df['signup_date'])
).astype('int8')

temporal_errors = int(df['temporal_inconsistency_flag'].sum())
print(f'Registros con última compra anterior al alta: {temporal_errors:,} ({temporal_errors/len(df):.2%})')

df['last_purchase_date_clean'] = df['last_purchase_date']
df.loc[df['temporal_inconsistency_flag'].eq(1), 'last_purchase_date_clean'] = pd.NaT


### Fecha de referencia

El dataset no trae una fecha explícita de extracción. Para crear variables relativas sin utilizar la fecha actual del equipo, se define una fecha de referencia reproducible:

**un día después de la fecha máxima válida observada de última compra.**

Esto simula un snapshot inmediatamente posterior al período observado y evita que la variable cambie cada vez que se ejecute el notebook.


In [ ]:
REFERENCE_DATE = df['last_purchase_date'].max() + pd.Timedelta(days=1)
print('Fecha de referencia:', REFERENCE_DATE.date())

df['customer_tenure_days'] = (REFERENCE_DATE - df['signup_date']).dt.days
df['recency_days'] = (REFERENCE_DATE - df['last_purchase_date_clean']).dt.days

assert (df['customer_tenure_days'].dropna() >= 0).all()
assert (df['recency_days'].dropna() >= 0).all()

display(df[['signup_date', 'last_purchase_date', 'last_purchase_date_clean', 'customer_tenure_days', 'recency_days', 'temporal_inconsistency_flag']].head())


## 6. Country y city: control de consistencia

In [ ]:
country_city = pd.crosstab(df['country'], df['city'])
display(country_city)

print('Número de países:', df['country'].nunique())
print('Número de ciudades:', df['city'].nunique())


La tabla muestra combinaciones país–ciudad incompatibles. No existe en el dataset una fuente que permita decidir si el dato erróneo es `country` o `city`. Por eso **no se corrigen artificialmente**.

Para el EDA se mantendrán ambas variables, dejando documentada la limitación. Para Machine Learning se comparará el desempeño con y sin `city`; si se excluye, será por calidad y no por conveniencia de métricas.

## 7. Controles automáticos de calidad posteriores

In [ ]:
quality_checks = {
    'filas_conservadas': len(df) == len(df_raw),
    'customer_id_unico': df['customer_id'].is_unique,
    'churn_sin_faltantes': df['churn'].notna().all(),
    'churn_binario': set(df['churn'].dropna().unique()).issubset({0, 1}),
    'sin_edades_menores_18': not (df['age'].dropna() < 18).any(),
    'tenure_no_negativo': (df['customer_tenure_days'].dropna() >= 0).all(),
    'recency_no_negativo': (df['recency_days'].dropna() >= 0).all(),
}

checks = pd.Series(quality_checks, name='OK')
display(checks.to_frame())

assert checks.all(), 'Falló al menos un control de calidad.'
print('Todos los controles de calidad pasaron.')


## 8. Resumen antes vs. después

In [ ]:
comparison = pd.DataFrame({
    'raw_missing': df_raw.isna().sum(),
    'clean_missing': df[df_raw.columns].isna().sum(),
})
comparison['delta_missing'] = comparison['clean_missing'] - comparison['raw_missing']
display(comparison[comparison[['raw_missing', 'clean_missing']].max(axis=1) > 0])

print(f'Columnas nuevas de calidad/feature engineering: {df.shape[1] - df_raw.shape[1]}')
print('Nuevas columnas:')
display(pd.Series([c for c in df.columns if c not in df_raw.columns], name='columna'))


## 9. Exportación

Se guarda una versión limpia para EDA y etapas posteriores.

> Nota: el archivo procesado puede regenerarse desde el CSV original y este notebook. Para los modelos, la imputación y codificación se implementarán dentro de pipelines reproducibles.


In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Archivo generado: {OUTPUT_PATH}')
print(f'Dimensiones: {df.shape}')


## Conclusión del paso de limpieza

- No se eliminaron clientes.
- Se corrigieron valores inválidos sin inventar información.
- Se conservaron flags de anomalías y faltantes.
- Las inconsistencias temporales quedaron explícitamente trazadas.
- Se generaron `customer_tenure_days` y `recency_days` con una fecha de referencia reproducible.
- La imputación estadística queda reservada al pipeline de Machine Learning para prevenir leakage.

### Próximo paso
`03_eda.ipynb`: análisis exploratorio univariado, bivariado y multivariado, con visualizaciones orientadas al problema de churn.
